<a href="https://colab.research.google.com/github/Brandeis-Visual-Analytics/COSI-165b-pas/blob/main/pa3/pa3_convolutional_networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COSI 165b - Deep Learning

Programming Assignment 3

Brandeis University

October 2026

**Check Moodle for Due Date**.  All assignments are due at 11:59 PM on the date listed on Moodle.  Check the syllabus for late submission policy.

**Any AI usage must be reported** at the [unique disclosure form for this PA3](https://genai.cs.brandeis.edu/prompt/cb9d5a3b-23a1-41aa-85c7-cf2e54d9f52f).  AI usage will not affect your grade; we are gathering this information to help us design future instances of the course.

## Convolutional networks: smaller, better, and why

**Name:** _your name_

This is an **individual** assignment. Make sure that you **File -> Save a copy in Drive** first (so your work persists).  Expected behavior for this notebook is that it runs **top to bottom** to produce all answers.  It may be helpful to read along **UDL Chapter 10** (Sections 10.1-10.3) and to look at the PyTorch tutorial [Training a Classifier](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html) as you complete this assignment.

**What you'll do:** In PA2 you trained fully connected networks on **KMNIST** and on **CIFAR-10**. Now you'll go back to **the same two datasets** with **convolutional** networks (`Conv2d` and pooling layers), and chase two goals on each:

- **Goal A - small:** find the **smallest** network (fewest parameters) that reaches the **accuracy your PA2 networks reached**. (On KMNIST that is about 92%, with a PA2 network of about 24,000 parameters. Can you match it with far fewer?)
- **Goal B - best:** find the **most accurate** network you can that stays **under a parameter cap**. The caps are chosen so that a network trains for 100 epochs in about 10 minutes or less on a free Colab GPU. A helper blocks training if your network is over the cap.

Then you'll **compare your results to fully connected networks (PA2)** and reflect on what changed and why.

**Runtime:** turn on a GPU now: *Runtime -> Change runtime type -> T4 GPU*. **Heads up: this assignment involves a lot of training** - you will train a dozen or more networks across two datasets, and the "best network" goals reward experimenting. Start early, explore with a small number of epochs (10-15), and save the long runs for your final candidates. `train()` prints an estimate of the run time after the first epoch; use it.

**Submitting:** *File -> Save a copy in Drive* first. Then *Share -> General access: Brandeis University, Viewer -> Copy link*, and paste the link into the PA3 assignment on Moodle. Run all cells so your outputs are visible.

## Part 0 - Setup
Run these cells. They define a **flexible `train()` loop** (similar to PA2's), a **parameter budget guard**, and a few helpers:

- `count_params(model)` - number of trainable parameters.
- `shape_trace(model, input_shape)` - prints the output shape after every layer of an `nn.Sequential`. **Use this a lot**: it's how you figure out the input size of the final `Linear` layer.
- `train(...)` - trains and returns a history. Mini-batches come from a **`DataLoader`** (as in PA2). If you pass `max_params=...`, it **prints your parameter count and refuses to train** (raises `ParamBudgetError`) if the model is over the cap. After the first epoch it **estimates how long the run will take** for the number of epochs you asked for and warns if that is over 10 minutes.
- `log_run(...)` / `show_results(...)` - keep a **table of every run** you do, so the notebook documents what you tried. **Log every run you want to count.**

**Read `train()` closely** - note how it uses the `DataLoader`, and that the training loss/accuracy it prints are running averages over each epoch (with dropout and augmentation switched on), while the validation numbers are measured in eval mode.

In [ ]:
import time, math, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)
print("device:", device)

# ---- The assignment's knobs (do not edit) ------------------------------------
PARAM_CAP_K = 25_000      # Goal B parameter cap for KMNIST (about the size of the PA2 MLPs)
PARAM_CAP_C = 250_000     # Goal B parameter cap for CIFAR-10
TARGET_ACC_K = 0.92       # Goal A baseline for KMNIST (validation accuracy; about what PA2 reached)
TARGET_ACC_C = 0.60       # Goal A baseline for CIFAR-10 (validation accuracy)
EPOCH_BUDGET = 100        # "100 epochs ..."
TIME_BUDGET_S = 600       # "... in under 10 minutes"

In [ ]:
class ParamBudgetError(RuntimeError):
    """Raised by train() when a model has more parameters than the allowed cap."""


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def check_budget(model, cap=None, label="model"):
    """Print the parameter count. If a cap is given and the model is over it, STOP."""
    n = count_params(model)
    if cap is None:
        print(f"[{label}] {n:,} parameters (no cap)")
        return n
    ok = n <= cap
    print(f"[{label}] {n:,} parameters   cap = {cap:,}   -> {'OK' if ok else 'OVER BUDGET'}")
    if not ok:
        raise ParamBudgetError(f"{label} has {n:,} parameters, which is {n - cap:,} over the cap of {cap:,}. "
                               "Shrink the network (fewer channels, a smaller Linear layer, more pooling) and try again.")
    return n


def shape_trace(model, input_shape):
    """Print the output shape after each layer of an nn.Sequential.
    input_shape excludes the batch dimension, e.g. (1, 28, 28) or (3, 32, 32)."""
    dev = next(model.parameters()).device
    x = torch.zeros(1, *input_shape, device=dev)
    model.eval()
    with torch.no_grad():
        print(f"input                -> {tuple(x.shape[1:])}")
        for i, layer in enumerate(model):
            x = layer(x)
            print(f"{i:2d} {layer.__class__.__name__:<15} -> {tuple(x.shape[1:])}")


def make_loader(X, y, batch_size, shuffle):
    """A standard PyTorch DataLoader over in-memory tensors: it batches (and, if shuffle=True, reshuffles every epoch)."""
    return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=shuffle, pin_memory=(device == "cuda"))


def evaluate(model, X, y, bs=1000):
    """Mean loss and accuracy over (X, y), in eval mode, using a DataLoader so big image sets fit in memory."""
    model.eval()
    sum_loss = nn.CrossEntropyLoss(reduction="sum")
    total, correct = 0.0, 0
    with torch.no_grad():
        for xb, yb in make_loader(X, y, bs, shuffle=False):
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            total += sum_loss(out, yb).item()
            correct += (out.argmax(1) == yb).sum().item()
    return total / X.shape[0], correct / X.shape[0]


def train(model, Xtr, ytr, Xval, yval, optimizer, epochs, batch_size=128,
          max_params=None, label="model", scheduler=None, augment_fn=None, print_every=10):
    """One flexible training loop for the whole assignment. Mini-batches come from a torch DataLoader.

      epochs      -> how many passes over the training set
      max_params  -> if given, print the parameter count and BLOCK training if it is over this cap
      scheduler   -> optional torch.optim.lr_scheduler object, stepped once per epoch
      augment_fn  -> optional function applied to each training mini-batch of inputs
      print_every -> print a progress line every this many epochs
    Returns a history dict (per-epoch train/val loss and accuracy, time, parameter count).
    train_loss / train_acc are running averages over the epoch's mini-batches (in training mode, so with dropout
    and augmentation active); val_loss / val_acc are measured on the validation set in eval mode after each epoch.
    After the first epoch it estimates how long THIS run will take (epochs x time of epoch 1) and warns if that is
    over 10 minutes. When a cap is given it also estimates the time for EPOCH_BUDGET (100) epochs, the Goal B budget.
    """
    check_budget(model, max_params, label)          # <-- blocks right here if over budget
    model.to(device)
    train_loader = make_loader(Xtr, ytr, batch_size, shuffle=True)
    loss_fn = nn.CrossEntropyLoss()
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        run_loss, run_correct, seen = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            if augment_fn is not None:
                xb = augment_fn(xb)
            optimizer.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            optimizer.step()
            run_loss += loss.item() * yb.shape[0]
            run_correct += (out.argmax(1) == yb).sum().item()
            seen += yb.shape[0]
        if scheduler is not None:
            scheduler.step()
        tl, ta = run_loss / seen, run_correct / seen
        vl, va = evaluate(model, Xval, yval)
        hist["train_loss"].append(tl); hist["train_acc"].append(ta)
        hist["val_loss"].append(vl);   hist["val_acc"].append(va)
        if epoch == 0:
            t1 = time.time() - t0
            print(f"  (first epoch took {t1:.1f}s -> about {t1 * epochs / 60:.1f} min for the {epochs} epochs in this run)")
            if t1 * epochs > TIME_BUDGET_S:
                print(f"  WARNING: that is over the {TIME_BUDGET_S // 60}-minute budget. Consider fewer epochs or a smaller/cheaper network.")
            if max_params is not None and epochs != EPOCH_BUDGET:
                print(f"  (at {EPOCH_BUDGET} epochs this network would take about {t1 * EPOCH_BUDGET / 60:.1f} min; the Goal B budget is {TIME_BUDGET_S // 60} min)")
                if t1 * EPOCH_BUDGET > TIME_BUDGET_S:
                    print(f"  WARNING: {EPOCH_BUDGET} epochs would be over the {TIME_BUDGET_S // 60}-minute Goal B budget. Consider a cheaper network.")
        if (epoch + 1) % print_every == 0 or epoch == 0 or epoch + 1 == epochs:
            print(f"  epoch {epoch + 1:3d}/{epochs}   train_loss {tl:.3f}   val_loss {vl:.3f}"
                  f"   train_acc {ta:.3f}   val_acc {va:.3f}")
    hist.update(time=time.time() - t0, params=count_params(model), label=label, cap=max_params)
    return hist


def score(hist):
    """Our noise-resistant score for a run: mean validation accuracy over the LAST 5 epochs."""
    return float(np.mean(hist["val_acc"][-5:]))


def plot_curves(hist, title=""):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(hist["train_loss"], label="train"); ax[0].plot(hist["val_loss"], label="val")
    ax[0].set(xlabel="epoch", ylabel="cross-entropy loss", title="Loss  " + title); ax[0].legend()
    ax[1].plot(hist["train_acc"], label="train"); ax[1].plot(hist["val_acc"], label="val")
    ax[1].set(xlabel="epoch", ylabel="accuracy", title="Accuracy  " + title); ax[1].legend()
    plt.tight_layout(); plt.show()


RESULTS = []
def log_run(hist, dataset, goal, note=""):
    """Record a run in the results table. dataset: 'kmnist' or 'cifar10'.
    goal: 'baseline' (the provided MLPs), 'A' (smallest), or 'B' (best under the cap)."""
    RESULTS.append(dict(dataset=dataset, goal=goal, run=hist["label"], params=hist["params"], cap=hist["cap"],
                        score=round(score(hist), 4), train_acc=round(hist["train_acc"][-1], 4),
                        epochs=len(hist["val_acc"]), time_s=round(hist["time"], 1), note=note))
    print(f"logged: {hist['label']}  ({hist['params']:,} params, score {score(hist):.3f})")


def show_results(dataset=None):
    """Show every logged run (optionally for one dataset), best score first."""
    df = pd.DataFrame(RESULTS)
    if len(df) == 0:
        print("nothing logged yet"); return
    if dataset is not None:
        df = df[df.dataset == dataset]
    display(df.sort_values("score", ascending=False).reset_index(drop=True))


def goal_a_status(dataset, target):
    """Goal A: the smallest logged network (any run) whose score reaches the target."""
    df = pd.DataFrame(RESULTS)
    df = df[(df.dataset == dataset) & (df.goal == "A") & (df.score >= target)] if len(df) else df
    if len(df) == 0:
        print(f"Goal A ({dataset}): no logged goal-A run has reached {target:.0%} yet."); return
    best = df.sort_values("params").iloc[0]
    print(f"Goal A ({dataset}): smallest network to reach {target:.0%} = {int(best.params):,} parameters  [{best.run}]  score {best.score:.3f}")


def goal_b_status(dataset, cap):
    """Goal B: the best-scoring logged network that was trained with the cap enforced."""
    df = pd.DataFrame(RESULTS)
    df = df[(df.dataset == dataset) & (df.goal == "B") & (df.cap == cap)] if len(df) else df
    if len(df) == 0:
        print(f"Goal B ({dataset}): no logged goal-B run (with max_params={cap:,}) yet."); return
    best = df.sort_values("score", ascending=False).iloc[0]
    print(f"Goal B ({dataset}): best score under {cap:,} params = {best.score:.3f}  [{best.run}, {int(best.params):,} params]")

## Part 1 - KMNIST with 2D convolutions

**KMNIST** is the dataset from PA2: 28x28 grayscale images of ten handwritten Japanese characters. We use **the same split as PA2**: 50,000 images for training and 10,000 for validation. The separate 10,000-image **test set** is held out - **don't touch it until the very end** (Part 3).

One difference from PA2: we **do not flatten** the images. Every model takes input of shape `(batch, 1, 28, 28)`: **1 channel, height 28, width 28**. We standardize with a single mean and standard deviation (PA2 standardized each pixel separately, which doesn't make sense for a convolution that treats all positions alike). **Switch to a GPU runtime now.**

In [ ]:
from torchvision import datasets

km = datasets.KMNIST('.', train=True, download=True)
km_te = datasets.KMNIST('.', train=False, download=True)
KM_CLASSES = km.classes

Xall = km.data.float() / 255.
mu_k, sd_k = Xall[10000:].mean(), Xall[10000:].std()          # one mean/std, from the TRAIN images only
prep_k = lambda x: ((x - mu_k) / sd_k).unsqueeze(1)           # (N, 28, 28) -> (N, 1, 28, 28): add the channel dimension
Xval_k, yval_k = prep_k(Xall[:10000]), km.targets[:10000]     # same held-out 10k as PA2
Xtr_k,  ytr_k  = prep_k(Xall[10000:]), km.targets[10000:]
Xte_k,  yte_k  = prep_k(km_te.data.float() / 255.), km_te.targets
print("KMNIST  train:", tuple(Xtr_k.shape), " val:", tuple(Xval_k.shape), " test:", tuple(Xte_k.shape))

In [ ]:
# A peek at the data: 8 examples per class.
fig, axes = plt.subplots(10, 8, figsize=(8, 10))
for c in range(10):
    ex = km.data[km.targets == c][:8]
    for j in range(8):
        axes[c, j].imshow(ex[j], cmap="gray"); axes[c, j].set_xticks([]); axes[c, j].set_yticks([])
    axes[c, 0].set_ylabel(KM_CLASSES[c], rotation=0, ha="right", va="center", fontsize=11)
fig.suptitle("KMNIST - 8 examples per class", y=1.005)
plt.tight_layout(); plt.show()

### 1.1 The fully connected baseline (provided)
Here is the **PA2 shallow MLP** on KMNIST: flatten, one hidden layer of 30 units (about 24,000 parameters), trained with SGD (lr 0.1, batch size 128, 15 epochs) - the run from Part 2 of PA2. Run it: this is the accuracy your CNNs should **match with far fewer parameters**, or **beat with about as many**. (Your number may differ slightly from PA2: the standardization is different, and training is random.)

In [ ]:
def make_mlp_k():
    return nn.Sequential(nn.Flatten(),
                         nn.Linear(784, 30), nn.ReLU(),
                         nn.Linear(30, 10))

mlp_k = make_mlp_k()
h = train(mlp_k, Xtr_k, ytr_k, Xval_k, yval_k, torch.optim.SGD(mlp_k.parameters(), lr=0.1),
          epochs=15, batch_size=128, label="PA2 MLP 30", print_every=5)
plot_curves(h, "KMNIST MLP")
log_run(h, "kmnist", "baseline")

PA2_KMNIST_VAL_ACC = None    # TODO: type in the best KMNIST validation accuracy you reached in YOUR PA2 (e.g. 0.92)

### 1.2 Your first convolutional network
Now **you** write the model. Define `make_cnn_k` below so that it returns an `nn.Sequential` with these layers, **in this order**:

1. `Conv2d` (1 input channel -> `channels` output channels) -> `ReLU` -> `MaxPool2d`
2. `Conv2d` (`channels` -> `2 * channels`) -> `ReLU` -> `MaxPool2d`
3. `Flatten`
4. `Linear` (to the 10 classes)

Reminders: use **padding** in the convolutions so that they keep the height and width the same (for an odd kernel size, `padding=kernel_size // 2` does that), and use a pooling window of 2, which halves the height and width. The hardest part is the `in_features` of the `Linear` layer: don't guess - either calculate the number of features yourself, or build the model without the `Linear` first (or put in a temporary value), run `shape_trace`, and read off the flattened size.

You haven't written these layers yet, so look at the PyTorch tutorial [Training a Classifier](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html) (the network definition) and the docs for [`nn.Conv2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html), [`nn.MaxPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) and [`nn.Linear`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html).

In [ ]:
def make_cnn_k(channels=8, kernel_size=3):
    # TODO: build and return the nn.Sequential described above.
    # Input: (batch, 1, 28, 28).  Output: (batch, 10).
    pass

cnn = make_cnn_k()
shape_trace(cnn, (1, 28, 28))
for name, p in cnn.named_parameters():
    print(f"{name:12s} {str(tuple(p.shape)):20s} {p.numel():7d}")
print("total:", count_params(cnn), "   (the PA2 MLP had", count_params(make_mlp_k()), ")")

### Q1. Count the parameters.
For the CNN above (with `channels=8, kernel_size=3`): work out **by hand** how many parameters each layer has and show that it matches the printout (for a convolution use $C_i \cdot C_o \cdot K^2 + C_o$). Then compare the **total** with the PA2 MLP in 1.1: which has more parameters, and which layer holds most of the CNN's parameters? (3-5 sentences.)

> _Your answer here._

In [ ]:
# Train your first CNN (provided template). Copy this cell for every configuration you try.
cnn = make_cnn_k(channels=8, kernel_size=3)
h = train(cnn, Xtr_k, ytr_k, Xval_k, yval_k, torch.optim.Adam(cnn.parameters(), lr=1e-3),
          epochs=15, batch_size=128, label="first CNN c=8 k=3", print_every=5)
plot_curves(h, "KMNIST first CNN")
log_run(h, "kmnist", "baseline", note="first CNN")

### 1.3 Goal A (KMNIST): the smallest network that matches PA2
**Find the CNN with the fewest parameters whose score reaches `TARGET_ACC_K` (92%, about what PA2's MLPs reached)** - the *score* is the mean validation accuracy over the last 5 epochs. Train each for up to 30 epochs.

You can change anything about the architecture: the number of channels, the kernel size, the number of layers, **stride** instead of (or in addition to) pooling, and the size/placement of the `Linear` layer. Think about where the parameters are (Q1!). A trick to try: `nn.AdaptiveAvgPool2d(1)` averages each channel down to a single number, which makes the `Linear` layer tiny. **Log every run with `goal="A"`.**

**How many layers? Some rules of thumb.**

- Build in **blocks**: one or two `Conv2d -> ReLU` layers followed by a `MaxPool2d(2)`. Each pooling step halves the height and width, so a 28x28 image can be pooled only about 4 times (28 -> 14 -> 7 -> 3 -> 1), and a 32x32 image about 5 times, before there is nothing left.
- A common pattern is to **increase the channels as the image shrinks** (e.g., 8 -> 16 -> 32): each pooling step cuts the number of positions by 4, so doubling the channels keeps the cost of each block roughly balanced.
- A reasonable place to start: **2 blocks for KMNIST, 3 blocks for CIFAR-10**. Add depth or width only if the **training** accuracy is still low (underfitting). If training accuracy is high but validation accuracy is lower, more layers won't fix it - you need regularization (dropout, weight decay, augmentation) or a smaller network.
- Two stacked 3x3 convolutions see the same patch of the image as one 5x5 convolution, with fewer parameters and an extra nonlinearity.
- The `Linear` layer after `Flatten` has (channels x height x width x 10) weights, so a big final feature map makes it expensive. Keep the last feature map small (3x3 or 4x4) with more pooling or a stride, or use `nn.AdaptiveAvgPool2d(1)`.

In [ ]:
# Goal A playground - copy this cell for each architecture you try.
def make_A_k():
    return nn.Sequential(
        # TODO: design a small CNN. Input is (batch, 1, 28, 28); output must be (batch, 10).
    )

net = make_A_k()
h = train(net, Xtr_k, ytr_k, Xval_k, yval_k, torch.optim.Adam(net.parameters(), lr=1e-3),
          epochs=30, batch_size=128, label="A: describe your net", print_every=10)
plot_curves(h, "Goal A")
log_run(h, "kmnist", "A", note="what you changed")

In [ ]:
goal_a_status("kmnist", TARGET_ACC_K)
show_results("kmnist")

### Q2. What made it small?
Describe your **smallest network that reached the baseline**: its architecture and parameter count, and how many times smaller it is than the PA2 MLP. Which choices **saved the most parameters** without losing accuracy (kernel size, channels, pooling/stride, size of the final layer), and which ones **hurt accuracy** when you shrank them? Refer to the runs in your table. (4-8 sentences.)

> _Your answer here._

### 1.4 Goal B (KMNIST): the best network under the parameter cap
Now go the other way: **get the highest score you can with at most `PARAM_CAP_K` (25,000) parameters** - about the size of the PA2 MLPs - trained for up to **100 epochs**. Pass `max_params=PARAM_CAP_K` to `train()` - it will **block** any network over the cap. Watch the time estimate printed after the first epoch: if 100 epochs would take more than 10 minutes, the network is too expensive. Everything else is fair game: depth, width, kernel sizes, learning rate, batch size, optimizer, weight decay, dropout, a learning-rate schedule, data augmentation (see `augment_fn`), and so on. **Log every run with `goal="B"`.**

*Tip: change one thing at a time, and use the 30-epoch runs to explore before committing to a 100-epoch run.*

In [ ]:
# Goal B playground - copy this cell for each configuration you try.
def make_B_k():
    return nn.Sequential(
        # TODO: design your best CNN. Input is (batch, 1, 28, 28); output must be (batch, 10).
    )

net = make_B_k()
h = train(net, Xtr_k, ytr_k, Xval_k, yval_k, torch.optim.Adam(net.parameters(), lr=1e-3),
          epochs=30, batch_size=128, max_params=PARAM_CAP_K, label="B: describe your net", print_every=10)
plot_curves(h, "Goal B")
log_run(h, "kmnist", "B", note="what you changed")

In [ ]:
goal_b_status("kmnist", PARAM_CAP_K)
show_results("kmnist")

### Q3. What moved the needle?
For your **best network under the cap**: what is its architecture, parameter count and score? How does its accuracy compare with the PA2 MLP **at about the same number of parameters**? Which hyperparameters had the **biggest effect** on validation accuracy, and which had little effect? Look at the train and validation curves of your best and a weaker run - what is the **train/validation gap**, and what did you do about it? (5-8 sentences. Refer to your table.)

> _Your answer here._

In [ ]:
# Keep your chosen networks under these names - Part 3 uses them.
# (Re-train them here if you need to, or assign the trained objects from above.)
small_cnn_k = None     # TODO: your Goal A network (trained)
best_cnn_k  = None     # TODO: your Goal B network (trained)

## Part 2 - CIFAR-10 with 2D convolutions

Now **CIFAR-10**: 32x32 **color** images in 10 classes. We use **exactly the same subsample as PA2** (1,000 training images and 400 validation images per class), so you can compare directly. A **test set** (200 images per class) is held out for Part 3 - **don't use it until then.**

Unlike PA2, we **do not flatten** the images: every model takes input of shape `(batch, 3, 32, 32)` (**3 channels**, height 32, width 32). We standardize each color channel. **Switch to a GPU runtime now.**

In [ ]:
!pip install -q datasets
from datasets import load_dataset

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

def stream_cifar(split, per_class):
    """Stream CIFAR-10 from the HuggingFace mirror, stopping once we have per_class images of every class."""
    buckets = {c: [] for c in range(10)}
    for ex in load_dataset("uoft-cs/cifar10", split=split, streaming=True):
        c = ex["label"]
        if len(buckets[c]) < per_class:
            buckets[c].append(np.asarray(ex["img"], dtype=np.uint8))          # (32, 32, 3) RGB
        if all(len(v) >= per_class for v in buckets.values()):
            break
    imgs = np.stack([im for c in range(10) for im in buckets[c]])
    labels = np.array([c for c in range(10) for _ in range(per_class)])
    return imgs, labels

N_TRAIN, N_VAL, N_TEST = 1000, 400, 200
imgs, labels = stream_cifar("train", N_TRAIN + N_VAL)
is_train = (np.arange(len(labels)) % (N_TRAIN + N_VAL)) < N_TRAIN              # same split as PA2
test_imgs, test_labels = stream_cifar("test", N_TEST)

to_tensor = lambda a: torch.tensor(a, dtype=torch.float32).permute(0, 3, 1, 2) / 255.     # (N,32,32,3) -> (N,3,32,32)
Xtr2, Xval2, Xte2 = to_tensor(imgs[is_train]), to_tensor(imgs[~is_train]), to_tensor(test_imgs)
ytr2, yval2, yte2 = (torch.tensor(labels[is_train]), torch.tensor(labels[~is_train]), torch.tensor(test_labels))

mu2 = Xtr2.mean(dim=(0, 2, 3), keepdim=True)          # one mean and std PER COLOR CHANNEL, from train only
sd2 = Xtr2.std(dim=(0, 2, 3), keepdim=True)
Xtr2, Xval2, Xte2 = [(x - mu2) / sd2 for x in (Xtr2, Xval2, Xte2)]
print("CIFAR  train:", tuple(Xtr2.shape), " val:", tuple(Xval2.shape), " test:", tuple(Xte2.shape))

In [ ]:
# A peek at the data: 8 examples per class.
raw_train, raw_lab = imgs[is_train], labels[is_train]
fig, axes = plt.subplots(10, 8, figsize=(9, 11))
for c in range(10):
    cls = raw_train[raw_lab == c]
    for j in range(8):
        axes[c, j].imshow(cls[j]); axes[c, j].set_xticks([]); axes[c, j].set_yticks([])
    axes[c, 0].set_ylabel(CLASS_NAMES[c], rotation=0, ha="right", va="center", fontsize=11)
fig.suptitle("CIFAR-10 - 8 examples per class", y=1.005)
plt.tight_layout(); plt.show()

### 2.1 The fully connected baseline (provided)
The **PA2 CIFAR-10 MLP** again, with the same optimizer and training length as PA2 (SGD, lr 0.05, batch size 128, 30 epochs). The `Flatten` inside the model does what PA2 did by hand. (Your number may differ slightly from PA2: here each color channel, rather than each pixel, is standardized, and training is random.)

In [ ]:
def make_mlp_2d():
    return nn.Sequential(nn.Flatten(),
                         nn.Linear(3072, 256), nn.ReLU(),
                         nn.Linear(256, 128), nn.ReLU(),
                         nn.Linear(128, 10))

mlp_2d = make_mlp_2d()
h = train(mlp_2d, Xtr2, ytr2, Xval2, yval2, torch.optim.SGD(mlp_2d.parameters(), lr=0.05),
          epochs=30, batch_size=128, label="PA2 MLP 256-128", print_every=5)
plot_curves(h, "CIFAR-10 MLP")
log_run(h, "cifar10", "baseline")

PA2_CIFAR_VAL_ACC = None    # TODO: type in the final validation accuracy you got for the CIFAR MLP in YOUR PA2 (e.g. 0.39)

### 2.2 Your first CIFAR-10 convolutional network
Same recipe as 1.2, now for color images. Define `make_cnn_2d` below so that it returns an `nn.Sequential` with these layers, **in this order**:

1. `Conv2d` (3 input channels -> `channels` output channels) -> `ReLU` -> `MaxPool2d`
2. `Conv2d` (`channels` -> `2 * channels`) -> `ReLU` -> `MaxPool2d`
3. `Flatten`
4. `Linear` (to the 10 classes)

Use 3x3 kernels, **padding** so the convolutions keep the height and width, and a pooling window of 2. The input is 32x32 instead of 28x28, so the `in_features` of the `Linear` layer will be different from the one you used for KMNIST: work it out or use `shape_trace`.

In [ ]:
def make_cnn_2d(channels=16):
    # TODO: build and return the nn.Sequential described above.
    # Input: (batch, 3, 32, 32).  Output: (batch, 10).
    pass

cnn2 = make_cnn_2d()
shape_trace(cnn2, (3, 32, 32))
for name, p in cnn2.named_parameters():
    print(f"{name:12s} {str(tuple(p.shape)):20s} {p.numel():7d}")
print("total:", count_params(cnn2), "   (the PA2 MLP had", count_params(make_mlp_2d()), ")")

In [ ]:
# Train your first 2D CNN (provided template). Copy this cell for every configuration you try.
cnn2 = make_cnn_2d(channels=16)
h = train(cnn2, Xtr2, ytr2, Xval2, yval2, torch.optim.Adam(cnn2.parameters(), lr=1e-3),
          epochs=30, batch_size=128, label="first CNN c=16", print_every=5)
plot_curves(h, "CIFAR-10 first CNN")
log_run(h, "cifar10", "baseline", note="first CNN")

### Q4. Convolution vs. PA2 - first look.
Compare your first CNN with the CIFAR-10 MLP (2.1), in **parameters**, **validation accuracy**, and the **shape of the curves** (smoothness, train/validation gap). Is the CNN better, and is it *also* much smaller? (3-6 sentences.)

> _Your answer here._

### 2.3 Goal A (CIFAR-10): the smallest network that reaches the baseline
**Find the CNN with the fewest parameters whose score reaches `TARGET_ACC_C` (60%)** - far above the ~40% of the PA2 MLP. Same rules as before. Think about the cost of each choice: the final `Linear` layer after a big flattened feature map is often where most parameters hide - what can you do about that (more pooling, stride, fewer channels at the end)? **Log every run with `goal="A"`.**

**Layer-count guidance for CIFAR-10:** start with **3 conv blocks** (each `Conv2d -> ReLU -> MaxPool2d`, channels increasing as the image shrinks, so the last feature map is 4x4) and the same rules of thumb as in Part 1. The images are bigger and harder than KMNIST, so a bit more depth and width pays off.

In [ ]:
# Goal A playground - copy this cell for each architecture you try.
def make_A_2d():
    return nn.Sequential(
        # TODO: design a small CNN. Input is (batch, 3, 32, 32); output must be (batch, 10).
    )

net = make_A_2d()
h = train(net, Xtr2, ytr2, Xval2, yval2, torch.optim.Adam(net.parameters(), lr=1e-3),
          epochs=40, batch_size=128, label="A: describe your net", print_every=10)
plot_curves(h, "Goal A")
log_run(h, "cifar10", "A", note="what you changed")

In [ ]:
goal_a_status("cifar10", TARGET_ACC_C)
show_results("cifar10")

### Q5. What made it small?
Describe your **smallest CIFAR-10 network that reached the baseline**: architecture and parameter count. Where were the parameters in your first CNN, and what did you change to cut them? How does the size compare with your smallest KMNIST network, and with the PA2 MLP? (Why is the CIFAR-10 network so much bigger?) (4-8 sentences.)

> _Your answer here._

### 2.4 Goal B (CIFAR-10): the best network under the parameter cap
**Get the highest score you can with at most `PARAM_CAP_C` (250,000) parameters**, trained for up to **100 epochs**. Pass `max_params=PARAM_CAP_C` to `train()` so oversize networks are blocked. Watch the time estimate printed after the first epoch: **if 100 epochs would take more than 10 minutes, the network is too expensive** (note that a network can be small in *parameters* but expensive in *computation* - e.g., many wide convolutions at full 32x32 resolution).

Everything is fair game: depth, width, kernel sizes, pooling vs. stride, `nn.Dropout`, weight decay, learning-rate schedules, batch size, `nn.BatchNorm2d` (we'll cover this next week - try it if you're curious), and data augmentation. Two augmentation functions are provided below to pass as `augment_fn`: `random_flip` and `random_crop` (random translation). To use both: `augment_fn=lambda xb: random_crop(random_flip(xb))`. With only 10,000 training images, augmentation is one of the biggest levers you have. **Log every run with `goal="B"`.**

In [ ]:
def random_flip(xb):
    """Randomly mirror each image in a (batch, 3, H, W) tensor left-to-right."""
    flip = torch.rand(xb.shape[0], device=xb.device) < 0.5
    return torch.where(flip[:, None, None, None], xb.flip(3), xb)


def random_crop(xb, pad=4):
    """Randomly translate each image: pad it with `pad` pixels of the mean color, then crop back to the original size
    at a random position (a different one for every image). Combine with random_flip by nesting the calls."""
    n, c, h, w = xb.shape
    xp = F.pad(xb, (pad, pad, pad, pad))                       # zeros = the mean color, since inputs are standardized
    i = torch.randint(0, 2 * pad + 1, (n,), device=xb.device)
    j = torch.randint(0, 2 * pad + 1, (n,), device=xb.device)
    rows = (i[:, None] + torch.arange(h, device=xb.device))     # (n, h): which padded rows to keep for each image
    cols = (j[:, None] + torch.arange(w, device=xb.device))     # (n, w)
    return xp[torch.arange(n, device=xb.device)[:, None, None, None], torch.arange(c, device=xb.device)[None, :, None, None],
              rows[:, None, :, None], cols[:, None, None, :]]

In [ ]:
# Goal B playground - copy this cell for each configuration you try.
def make_B_2d():
    return nn.Sequential(
        # TODO: design your best CNN. Input is (batch, 3, 32, 32); output must be (batch, 10).
    )

net = make_B_2d()
h = train(net, Xtr2, ytr2, Xval2, yval2, torch.optim.Adam(net.parameters(), lr=1e-3),
          epochs=100, batch_size=128, max_params=PARAM_CAP_C, label="B: describe your net", print_every=10,
          augment_fn=None)       # try augment_fn=random_flip, random_crop, or both
plot_curves(h, "Goal B")
log_run(h, "cifar10", "B", note="what you changed")

In [ ]:
goal_b_status("cifar10", PARAM_CAP_C)
show_results("cifar10")

### Q6. What moved the needle on CIFAR-10?
For your **best network under the cap**: architecture, parameter count, score, and training time. Which hyperparameters (and which architecture choices) made the biggest difference? Is it overfitting - how large is the train/validation gap, and which of your changes (dropout, weight decay, augmentation, ...) helped or didn't? (5-8 sentences. Refer to your table.)

> _Your answer here._

In [ ]:
# Keep your chosen networks under these names - Part 3 uses them.
small_cnn_2d = None     # TODO: your Goal A network (trained)
best_cnn_2d  = None     # TODO: your Goal B network (trained)

## Part 3 - Compare with PA2, and reflect

### 3.1 One table
Run this cell to put the MLPs and your CNNs side by side (it uses your logged runs). Then, **once**, evaluate your final networks on the **held-out test sets**. Compare each test accuracy with its validation accuracy - they will not be equal, especially on KMNIST.

In [ ]:
rows = []
for ds, cap, target in [("kmnist", PARAM_CAP_K, TARGET_ACC_K), ("cifar10", PARAM_CAP_C, TARGET_ACC_C)]:
    df = pd.DataFrame(RESULTS); df = df[df.dataset == ds]
    for _, r in df[df.goal == "baseline"].iterrows():
        if r.run.startswith("PA2"):
            rows.append(dict(dataset=ds, model="fully connected (PA2)", params=int(r.params), val_score=r.score, train_acc=r.train_acc))
    a = df[(df.goal == "A") & (df.score >= target)]
    if len(a): r = a.sort_values("params").iloc[0]; rows.append(dict(dataset=ds, model="CNN, goal A (smallest)", params=int(r.params), val_score=r.score, train_acc=r.train_acc))
    b = df[(df.goal == "B") & (df.cap == cap)]
    if len(b): r = b.sort_values("score", ascending=False).iloc[0]; rows.append(dict(dataset=ds, model="CNN, goal B (best under cap)", params=int(r.params), val_score=r.score, train_acc=r.train_acc))
display(pd.DataFrame(rows))
print("PA2 validation accuracy (yours, from PA2):  KMNIST", PA2_KMNIST_VAL_ACC, "  CIFAR-10", PA2_CIFAR_VAL_ACC)

In [ ]:
# Final test-set accuracy - run this ONCE, after you are done tuning.
def test_acc(model, X, y):
    return evaluate(model, X, y)[1]

for name, model, X, y in [("KMNIST MLP",        mlp_k,       Xte_k, yte_k),
                          ("KMNIST goal A CNN", small_cnn_k, Xte_k, yte_k),
                          ("KMNIST goal B CNN", best_cnn_k,  Xte_k, yte_k),
                          ("CIFAR-10 MLP",        mlp_2d,       Xte2, yte2),
                          ("CIFAR-10 goal A CNN", small_cnn_2d, Xte2, yte2),
                          ("CIFAR-10 goal B CNN", best_cnn_2d,  Xte2, yte2)]:
    print(f"{name:22s} params {count_params(model):>9,}   test accuracy {test_acc(model, X, y):.3f}")

### Q7. Why do convolutional networks do better?
Using your numbers (parameters, validation and test accuracy, train/validation gap) from **both** datasets and **PA2**: how did the CNNs compare with the fully connected networks, and **what is the mechanism**? Use the vocabulary from lecture - **local connections, weight sharing, parameter count, inductive bias** - and say which of these your results actually support. On KMNIST the story was mostly *"the same accuracy with far fewer parameters"*; on CIFAR-10 it was *"much better accuracy"*. Why might the benefit of convolution show up differently on the two datasets? Finally, compare the **test** accuracies with the validation accuracies: is there a gap, and does the CNN's advantage over the MLP hold up on the test set? (6-10 sentences.)

> _Your answer here._

### 3.2 A test of translation
Convolution is **equivariant** to translation, and pooling plus the final layers can make a classifier approximately **invariant**. Does that show up in practice? The helper below **shifts** every image by `dx` pixels (horizontally; the image wraps around) and measures accuracy. Run it for the KMNIST MLP and your best KMNIST CNN. Then **do the same for CIFAR-10**, shifting images horizontally by 0 to 8 pixels.

In [ ]:
def shifted(X, dx):
    return torch.roll(X, shifts=dx, dims=-1)

shifts = list(range(-6, 7, 2))
acc_mlp = [evaluate(mlp_k,      shifted(Xval_k, dx), yval_k)[1] for dx in shifts]
acc_cnn = [evaluate(best_cnn_k, shifted(Xval_k, dx), yval_k)[1] for dx in shifts]
plt.figure(figsize=(6, 3.8))
plt.plot(shifts, acc_mlp, "o-", label="MLP"); plt.plot(shifts, acc_cnn, "o-", label="best CNN")
plt.xlabel("horizontal shift of the image (pixels)"); plt.ylabel("validation accuracy"); plt.title("KMNIST: accuracy vs. shift"); plt.legend(); plt.show()

In [ ]:
# TODO: the same experiment for CIFAR-10. Shift Xval2 horizontally by 0, 2, 4, 6, 8 pixels (try negative too)
# and plot accuracy vs. shift for mlp_2d and best_cnn_2d.
# Your code here

### Q8. What did shifting reveal?
Describe how accuracy changed with shift for the MLPs and the CNNs on both datasets. Which network was **more robust**, and why - in terms of **equivariance**, **invariance**, and what pooling does? Was your CNN **perfectly** invariant? Why or why not? (4-8 sentences.)

> _Your answer here._

### Q9. Reflection.
Look back at the whole assignment (PA2 -> PA3). In **one short paragraph (5-8 sentences)**: (a) What was the most surprising thing you saw? (b) Which single hyperparameter or design choice mattered most, and for which dataset? (c) If you had another week, what would you try next to improve your best CIFAR-10 network, and why do you expect it to help?

> _Your answer here._